# Phase 2 L-NER - Experiment 1: Full Fine-tuning


Auto-finds `best.pt`, `bpe_40k.json`, `a1_tokenizer.py` from Google Drive.

In [ ]:
# Mount Google Drive (Colab)
from google.colab import drive
drive.mount('/content/drive')

## 1 - Imports & Setup

In [ ]:
import os, sys, re, json, math, time, random, bisect, copy, zipfile, glob, shutil
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

print("Imports OK")
print("PyTorch:", torch.__version__)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", f"{torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2 - Configuration

In [ ]:
@dataclass
class NERConfig:
    # ── Model (must match Phase 1) ──
    vocab_size: int      = 40_000
    d_model: int         = 768
    n_layers: int        = 19
    n_query_heads: int   = 12
    n_kv_heads: int      = 4
    d_ff: int            = 2048
    max_seq_len: int     = 512
    dropout: float       = 0.1
    weight_tying: bool   = True
    rope_theta: float    = 10000.0

    # ── NER ──
    num_labels: int      = 12
    bio_num_tags: int    = 25      # 2*12 + 1 (O)
    window_size: int     = 510
    window_stride: int   = 255     # 50% overlap

    # ── Training ──
    lr: float            = 2e-5
    min_lr: float        = 1e-6
    warmup_steps: int    = 50
    num_epochs: int      = 20
    batch_size: int      = 8
    grad_accum: int      = 2       # effective batch = 16
    weight_decay: float  = 0.01
    max_grad_norm: float = 1.0
    use_amp: bool        = True
    seed: int            = 42

    def __post_init__(self):
        self.head_dim = self.d_model // self.n_query_heads
        self.n_groups = self.n_query_heads // self.n_kv_heads

CFG = NERConfig()

OUT       = Path("/content")
CKPT_DIR  = OUT / "ner_checkpoints"; CKPT_DIR.mkdir(exist_ok=True)
SUB_DIR   = OUT / "submissions";     SUB_DIR.mkdir(exist_ok=True)

print(f"Config: {CFG.d_model}d x {CFG.n_layers}L, window={CFG.window_size}, stride={CFG.window_stride}")

## 3 - Find Files & Load Tokenizer

In [ ]:
# ── Auto-find files in Google Drive ──
DRIVE_ROOT = "/content/drive/MyDrive"

def find_file(filename):
    matches = glob.glob(os.path.join(DRIVE_ROOT, "**", filename), recursive=True)
    if matches:
        print(f"  Found {filename}: {matches[0]}")
        return matches[0]
    raise FileNotFoundError(f"{filename} not found in Google Drive!")

def find_checkpoint():
    # best.pt first
    matches = glob.glob(os.path.join(DRIVE_ROOT, "**", "best.pt"), recursive=True)
    if matches:
        print(f"  Found best.pt: {matches[0]}")
        return matches[0]
    # highest step_*.pt
    all_steps = []
    for match in glob.glob(os.path.join(DRIVE_ROOT, "**", "step_*.pt"), recursive=True):
        base = os.path.basename(match)
        try:
            step_num = int(base.replace("step_", "").replace(".pt", ""))
            all_steps.append((step_num, match))
        except ValueError:
            continue
    if all_steps:
        all_steps.sort(reverse=True)
        print(f"  Found highest checkpoint: {all_steps[0][1]}")
        return all_steps[0][1]
    raise FileNotFoundError("No checkpoint found in Google Drive!")

PHASE1_CKPT = find_checkpoint()
tok_json_path = find_file("bpe_40k.json")
tok_py_path = find_file("a1_tokenizer.py")

shutil.copy(tok_json_path, "/content/bpe_40k.json")
shutil.copy(tok_py_path, "/content/a1_tokenizer.py")
sys.path.insert(0, "/content")

from a1_tokenizer import BPETokenizer
tokenizer = BPETokenizer.load("/content/bpe_40k.json")
CFG.vocab_size = tokenizer.vocab_size
print(f"Tokenizer loaded: vocab_size={tokenizer.vocab_size}")

# Smoke test
_t = "The Supreme Court held that"
assert tokenizer.decode(tokenizer.encode(_t)) == _t, "Round-trip failed"
print("Tokenizer OK")

## 4 - Download NER Data

In [ ]:
# ── Download NER Data from HuggingFace ──
!pip -q install huggingface_hub
from huggingface_hub import hf_hub_download

# You need a HF token with access to the dataset
# Set it here or use Colab secrets
HF_REPO = "Exploration-Lab/CS-781-Capstone"
HF_TOKEN = ""  # <-- paste your HuggingFace token here

# If using Colab secrets:
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    print("HF token loaded from Colab secrets")
except:
    if not HF_TOKEN:
        print("WARNING: Set HF_TOKEN above or add it to Colab secrets!")

NER_FILES = {
    "fold1_text":  "Phase_2_LNER/Train/fold1-text.json",
    "fold1_gold":  "Phase_2_LNER/Train/fold1-gold.json",
    "fold2_text":  "Phase_2_LNER/Train/fold2-text.json",
    "fold2_gold":  "Phase_2_LNER/Train/fold2-gold.json",
    "labels":      "Phase_2_LNER/Train/labels.txt",
    "fold3_text":  "Phase_2_LNER/Test/fold3-text.json",
}

ner_paths = {}
for key, fn in NER_FILES.items():
    ner_paths[key] = hf_hub_download(HF_REPO, fn, repo_type="dataset", token=HF_TOKEN)
    print(f"  {key:12s} -> downloaded")

## 5 - Parse Data & Label Maps

In [ ]:
# ── Parse Data & Build Label Maps ──
with open(ner_paths["labels"], "r") as f:
    LABELS = [l.strip() for l in f if l.strip()]
print(f"Entity labels ({len(LABELS)}): {LABELS}")

O_TAG = 0
tag2id = {"O": O_TAG}
id2tag = {O_TAG: "O"}
for i, lab in enumerate(LABELS):
    tag2id[lab] = i + 1
    id2tag[i + 1] = lab

CFG.num_labels = len(LABELS)
CFG.bio_num_tags = len(tag2id)   # now 13 instead of 25
print(f"Flat tags: {CFG.bio_num_tags} ({len(LABELS)} labels + O)")

def load_json(p):
    with open(p, "r") as f:
        return json.load(f)

fold1_text = load_json(ner_paths["fold1_text"])
fold1_gold = load_json(ner_paths["fold1_gold"])
fold2_text = load_json(ner_paths["fold2_text"])
fold2_gold = load_json(ner_paths["fold2_gold"])
fold3_text = load_json(ner_paths["fold3_text"])

print(f"Fold 1: {len(fold1_text)} docs")
print(f"Fold 2: {len(fold2_text)} docs")
print(f"Fold 3 (test): {len(fold3_text)} docs")


## 6 - Offset-Tracking Encoder

In [ ]:
# ── Offset-Tracking Encoder ──
def encode_with_offsets(tokenizer, text):
    token_ids = []
    char_spans = []
    for m in re.finditer(r'\S+', text):
        word = m.group()
        ws = m.start()
        char_byte_bounds = [0]
        for ch in word:
            char_byte_bounds.append(char_byte_bounds[-1] + len(ch.encode('utf-8')))
        bpe_syms = tokenizer._bpe(word)
        SPACE_BYTES = 1
        fb_pos = 0
        for sym in bpe_syms:
            tok_id = tokenizer.token_to_id.get(sym, tokenizer.token_to_id.get('<unk>', 1))
            n_bytes = len(sym)
            fb_end = fb_pos + n_bytes
            wb_s = max(0, fb_pos - SPACE_BYTES)
            wb_e = max(0, fb_end - SPACE_BYTES)
            wb_e = min(wb_e, char_byte_bounds[-1])
            wb_s = min(wb_s, char_byte_bounds[-1])
            if wb_e > wb_s:
                c_s = bisect.bisect_right(char_byte_bounds, wb_s) - 1
                c_e = bisect.bisect_left(char_byte_bounds, wb_e)
                c_s = max(0, c_s)
                c_e = min(len(word), c_e)
                char_spans.append((ws + c_s, ws + c_e))
            else:
                char_spans.append((ws, ws))
            token_ids.append(tok_id)
            fb_pos = fb_end
    return token_ids, char_spans

def get_doc_text(doc_entry):
    if isinstance(doc_entry, str):
        return doc_entry
    if isinstance(doc_entry, dict):
        return doc_entry.get("text", "")
    return str(doc_entry)

print("Offset encoder defined")

## 7 - Model Architecture

In [ ]:
# ── Phase 1 Model Architecture ──
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        rms = torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() * rms).to(x.dtype) * self.weight

def precompute_rope(dim, max_len, theta=10000.0, device=None):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2, device=device).float() / dim))
    t = torch.arange(max_len, device=device).float()
    angles = torch.outer(t, freqs)
    return angles.cos(), angles.sin()

def apply_rope(x, cos, sin, offset=0):
    T = x.size(2)
    c = cos[offset:offset+T].unsqueeze(0).unsqueeze(0)
    s = sin[offset:offset+T].unsqueeze(0).unsqueeze(0)
    hd = x.size(-1) // 2
    x1, x2 = x[..., :hd], x[..., hd:]
    rotated = torch.cat([-x2, x1], dim=-1)
    c_exp = torch.cat([c.expand(-1,-1,-1,hd), c.expand(-1,-1,-1,hd)], dim=-1)
    s_exp = torch.cat([s.expand(-1,-1,-1,hd), s.expand(-1,-1,-1,hd)], dim=-1)
    return x * c_exp + rotated * s_exp

class GQA(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.nqh = cfg.n_query_heads
        self.nkh = cfg.n_kv_heads
        self.ng  = cfg.n_groups
        self.hd  = cfg.head_dim
        self.dm  = cfg.d_model
        self.q = nn.Linear(cfg.d_model, cfg.n_query_heads * cfg.head_dim, bias=False)
        self.k = nn.Linear(cfg.d_model, cfg.n_kv_heads   * cfg.head_dim, bias=False)
        self.v = nn.Linear(cfg.d_model, cfg.n_kv_heads   * cfg.head_dim, bias=False)
        self.o = nn.Linear(cfg.n_query_heads * cfg.head_dim, cfg.d_model, bias=False)
        self.attn_drop  = nn.Dropout(cfg.dropout)
        self.resid_drop = nn.Dropout(cfg.dropout)
        self.scale = 1.0 / math.sqrt(cfg.head_dim)

    def forward(self, x, cos, sin, mask=None, kv_cache=None, offset=0):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.nqh, self.hd).transpose(1,2)
        k = self.k(x).view(B, T, self.nkh, self.hd).transpose(1,2)
        v = self.v(x).view(B, T, self.nkh, self.hd).transpose(1,2)
        q = apply_rope(q, cos, sin, offset)
        k = apply_rope(k, cos, sin, offset)
        if kv_cache is not None:
            k = torch.cat([kv_cache[0], k], dim=2)
            v = torch.cat([kv_cache[1], v], dim=2)
        new_cache = (k, v)
        if self.ng > 1:
            k = k.unsqueeze(2).expand(B, self.nkh, self.ng, k.size(2), self.hd).reshape(B, self.nqh, -1, self.hd)
            v = v.unsqueeze(2).expand(B, self.nkh, self.ng, v.size(2), self.hd).reshape(B, self.nqh, -1, self.hd)
        w = torch.matmul(q, k.transpose(-2,-1)) * self.scale
        if mask is not None: w = w + mask
        w = F.softmax(w, dim=-1, dtype=torch.float32).to(q.dtype)
        w = self.attn_drop(w)
        out = torch.matmul(w, v).transpose(1,2).contiguous().view(B, T, self.dm)
        return self.resid_drop(self.o(out)), new_cache

class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.up   = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.down = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)
        self.drop = nn.Dropout(cfg.dropout)
    def forward(self, x):
        return self.drop(self.down(F.silu(self.gate(x)) * self.up(x)))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model)
        self.attn  = GQA(cfg)
        self.norm2 = RMSNorm(cfg.d_model)
        self.ffn   = SwiGLU(cfg)
    def forward(self, x, cos, sin, mask=None, kv_cache=None, offset=0):
        h, cache = self.attn(self.norm1(x), cos, sin, mask, kv_cache, offset)
        x = x + h
        x = x + self.ffn(self.norm2(x))
        return x, cache

class LegalLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb  = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop     = nn.Dropout(cfg.dropout)
        self.layers   = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.norm     = RMSNorm(cfg.d_model)
        self.lm_head  = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.weight_tying:
            self.lm_head.weight = self.tok_emb.weight
        cos, sin = precompute_rope(cfg.head_dim, cfg.max_seq_len * 2, cfg.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def _causal_mask(self, T, device, S=None):
        if S is None: S = T
        m = torch.full((T, S), float("-inf"), device=device)
        off = S - T
        for i in range(T):
            m[i, :off+i+1] = 0.0
        return m.unsqueeze(0).unsqueeze(0)

    def forward(self, ids, targets=None, kv_caches=None, offset=0):
        B, T = ids.shape
        x = self.drop(self.tok_emb(ids))
        mask = self._causal_mask(T, ids.device, T + offset)
        new_caches = []
        for i, layer in enumerate(self.layers):
            c = kv_caches[i] if kv_caches else None
            x, nc = layer(x, self.rope_cos, self.rope_sin, mask, c, offset)
            new_caches.append(nc)
        logits = self.lm_head(self.norm(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, self.cfg.vocab_size),
                                   targets.view(-1), ignore_index=-1)
        return logits, loss, new_caches

print("Model architecture defined")

## 8 - Load Phase 1 Backbone

In [ ]:
# ── Load Phase 1 Backbone ──
def load_phase1_backbone(cfg, ckpt_path, device):
    model = LegalLM(cfg)
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state = ckpt.get("model", ckpt)
    model.load_state_dict(state, strict=True)
    model = model.to(device)
    print(f"Phase 1 backbone loaded from {os.path.basename(ckpt_path)}")
    print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
    return model

print("Backbone loader defined")

## 9 - NER Model Wrapper

In [ ]:
# ── NER Model Wrapper ──
class LegalNER(nn.Module):
    def __init__(self, backbone, num_tags, dropout=0.3):
        super().__init__()
        self.backbone = backbone
        self.cfg = backbone.cfg
        self.num_tags = num_tags
        self.ner_dropout = nn.Dropout(dropout)
        self.ner_head = nn.Linear(self.cfg.d_model, num_tags)
        nn.init.normal_(self.ner_head.weight, std=0.02)
        nn.init.zeros_(self.ner_head.bias)

    def get_hidden_states(self, ids, attention_mask=None):
        B, T = ids.shape
        x = self.backbone.drop(self.backbone.tok_emb(ids))
        # Build additive attention mask: 0 = attend, -inf = block
        if attention_mask is not None:
            # attention_mask: [B, T] with 1=real, 0=pad
            mask = attention_mask.float()
            mask = mask[:, None, None, :]          # [B, 1, 1, T]
            mask = (1.0 - mask) * float("-inf")    # 0 -> -inf, 1 -> 0
        else:
            mask = None  # fully bidirectional, no padding
        for layer in self.backbone.layers:
            x, _ = layer(x, self.backbone.rope_cos, self.backbone.rope_sin, mask)
        x = self.backbone.norm(x)
        return x

    def forward(self, input_ids, labels=None, attention_mask=None):
        hidden = self.get_hidden_states(input_ids, attention_mask=attention_mask)
        hidden = self.ner_dropout(hidden)
        logits = self.ner_head(hidden)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits.view(-1, self.num_tags),
                labels.view(-1),
                ignore_index=-100,
            )
        return {"loss": loss, "logits": logits}

print(f"LegalNER: backbone -> {CFG.d_model}d -> {CFG.bio_num_tags} BIO tags")


## 10 - NER Dataset (Sliding Windows)

In [ ]:
# ── Flat Tagging & NER Dataset ──
def assign_bio_tags(char_spans, gold_annotations, tag2id):
    """Assign flat entity tags (no B/I prefix) to each token."""
    n_tokens = len(char_spans)
    tags = [O_TAG] * n_tokens
    golds = sorted(gold_annotations, key=lambda a: int(a[0]))
    for ann in golds:
        gs, ge, label = int(ann[0]), int(ann[1]), ann[2]
        if label not in LABELS:
            continue
        label_tag = tag2id.get(label, O_TAG)
        for t_idx, (cs, ce) in enumerate(char_spans):
            if ce <= cs:
                continue
            if cs < ge and ce > gs:
                tags[t_idx] = label_tag
    return tags


class NERDataset(Dataset):
    def __init__(self, text_dict, gold_dict, tokenizer, cfg, tag2id):
        self.windows = []
        self.cfg = cfg
        for doc_id in tqdm(text_dict, desc="Building NER dataset"):
            text = get_doc_text(text_dict[doc_id])
            if not text.strip():
                continue
            all_ids, all_spans = encode_with_offsets(tokenizer, text)
            n = len(all_ids)
            gold_anns = gold_dict.get(doc_id, [])
            all_tags = assign_bio_tags(all_spans, gold_anns, tag2id)
            if n == 0:
                continue
            step = cfg.window_stride if cfg.window_stride > 0 else cfg.window_size
            starts = list(range(0, max(1, n - cfg.window_size + 1), step))
            if starts[-1] + cfg.window_size < n:
                starts.append(n - cfg.window_size)
            for w_idx, s in enumerate(starts):
                e = min(s + cfg.window_size, n)
                w_ids = all_ids[s:e]
                w_tags = all_tags[s:e]
                self.windows.append((w_ids, w_tags, doc_id, w_idx))
        print(f"  {len(self.windows)} windows from {len(text_dict)} docs")

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        w_ids, w_tags, doc_id, w_idx = self.windows[idx]
        L = self.cfg.window_size
        pad_len = L - len(w_ids)
        ids = w_ids + [0] * pad_len
        tags = w_tags + [-100] * pad_len
        mask = [1] * len(w_ids) + [0] * pad_len
        return {
            "input_ids": torch.tensor(ids, dtype=torch.long),
            "labels":    torch.tensor(tags, dtype=torch.long),
            "attention_mask": torch.tensor(mask, dtype=torch.long),
        }

print("NERDataset defined (flat labels, no B/I)")


In [ ]:
# ── Combine both folds & split 90/10 train/val ──
all_text = {**fold1_text, **fold2_text}
all_gold = {**fold1_gold, **fold2_gold}
doc_ids = sorted(all_text.keys())
random.seed(CFG.seed)
random.shuffle(doc_ids)
split_idx = int(len(doc_ids) * 0.9)  # 63 train, 7 val
train_ids = doc_ids[:split_idx]
val_ids = doc_ids[split_idx:]
train_text = {d: all_text[d] for d in train_ids}
train_gold = {d: all_gold[d] for d in train_ids}
val_text = {d: all_text[d] for d in val_ids}
val_gold = {d: all_gold[d] for d in val_ids}
print(f"Train: {len(train_text)} docs, Val: {len(val_text)} docs")

print("Building train dataset ...")
train_ds = NERDataset(train_text, train_gold, tokenizer, CFG, tag2id)
print("Building val dataset ...")
val_ds   = NERDataset(val_text, val_gold, tokenizer, CFG, tag2id)

train_dl = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True,
                      num_workers=2, pin_memory=True, drop_last=True)
val_dl   = DataLoader(val_ds, batch_size=CFG.batch_size, shuffle=False,
                      num_workers=2, pin_memory=True)

print(f"Train: {len(train_ds)} windows, Val: {len(val_ds)} windows")



## 11 - Training & Evaluation

In [ ]:
# ── Training & Evaluation Functions ──
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def get_cosine_lr(step, total_steps, lr, min_lr, warmup_steps):
    if step < warmup_steps:
        return lr * (step + 1) / warmup_steps
    prog = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return min_lr + (lr - min_lr) * 0.5 * (1.0 + math.cos(math.pi * prog))

def train_ner(model, train_dl, val_dl, cfg, exp_name, device=DEVICE, patience=5):
    set_seed(cfg.seed)
    model = model.to(device)
    trainable = [p for p in model.parameters() if p.requires_grad]
    n_trainable = sum(p.numel() for p in trainable)
    n_total = sum(p.numel() for p in model.parameters())
    print(f"\n{'='*60}")
    print(f"Experiment: {exp_name}")
    print(f"Trainable: {n_trainable:,} / {n_total:,} ({100*n_trainable/n_total:.1f}%)")
    print(f"{'='*60}")

    optimizer = torch.optim.AdamW(trainable, lr=cfg.lr, weight_decay=cfg.weight_decay)
    scaler = torch.amp.GradScaler('cuda', enabled=cfg.use_amp)
    total_steps = len(train_dl) * cfg.num_epochs // cfg.grad_accum
    best_f1 = 0.0
    best_state = None
    global_step = 0
    no_improve = 0  # early stopping counter

    for epoch in range(cfg.num_epochs):
        model.train()
        running_loss = 0.0
        t0 = time.time()
        for batch_idx, batch in enumerate(train_dl):
            ids  = batch["input_ids"].to(device)
            labs = batch["labels"].to(device)
            amask = batch["attention_mask"].to(device)
            with torch.amp.autocast('cuda', enabled=cfg.use_amp):
                out = model(ids, labels=labs, attention_mask=amask)
                loss = out["loss"] / cfg.grad_accum
            scaler.scale(loss).backward()
            running_loss += loss.item()
            if (batch_idx + 1) % cfg.grad_accum == 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(trainable, cfg.max_grad_norm)
                lr = get_cosine_lr(global_step, total_steps, cfg.lr, cfg.min_lr, cfg.warmup_steps)
                for pg in optimizer.param_groups:
                    pg["lr"] = lr
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                global_step += 1
            if (batch_idx + 1) % (50 * cfg.grad_accum) == 0:
                avg = running_loss / (batch_idx + 1)
                elapsed = time.time() - t0
                print(f"  [{exp_name}] Epoch {epoch+1} step {batch_idx+1}/{len(train_dl)} | "
                      f"loss {avg:.4f} | lr {lr:.2e} | {elapsed:.0f}s")
        val_loss, macro_f1, per_label = evaluate_ner(model, val_dl, device)
        print(f"  [{exp_name}] Epoch {epoch+1} | val_loss={val_loss:.4f} | macro_f1={macro_f1:.4f}")
        if macro_f1 > best_f1:
            best_f1 = macro_f1
            best_state = copy.deepcopy(model.state_dict())
            ckpt_path = CKPT_DIR / f"{exp_name}_best.pt"
            torch.save({"model": best_state, "f1": best_f1, "epoch": epoch+1}, ckpt_path)
            print(f"  New best F1={best_f1:.4f} saved to {ckpt_path.name}")
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"  Early stopping at epoch {epoch+1} (no improvement for {patience} epochs)")
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    print(f"  [{exp_name}] Training done. Best macro-F1: {best_f1:.4f}")
    return model

@torch.no_grad()
def evaluate_ner(model, dl, device=DEVICE):
    model.eval()
    total_loss = 0
    n_batches = 0
    all_preds = []
    all_golds = []
    for batch in dl:
        ids  = batch["input_ids"].to(device)
        labs = batch["labels"].to(device)
        mask = batch["attention_mask"].to(device)
        with torch.amp.autocast('cuda', enabled=CFG.use_amp):
            out = model(ids, labels=labs, attention_mask=mask)
        total_loss += out["loss"].item()
        n_batches += 1
        preds = out["logits"].argmax(dim=-1)
        for b in range(ids.size(0)):
            length = mask[b].sum().item()
            p_seq = preds[b, :length].cpu().tolist()
            g_seq = labs[b, :length].cpu().tolist()
            all_preds.append(p_seq)
            all_golds.append(g_seq)
    macro_f1, per_label = compute_span_macro_f1(all_preds, all_golds)
    avg_loss = total_loss / max(n_batches, 1)
    return avg_loss, macro_f1, per_label

def compute_span_macro_f1(all_pred_seqs, all_gold_seqs):
    """Compute macro F1 using flat label span extraction."""
    def flat_to_spans(tag_seq):
        spans = []
        current_label = None
        current_start = None
        for i, tag_id in enumerate(tag_seq):
            if tag_id == -100:
                continue
            tag_name = id2tag.get(tag_id, "O")
            if tag_name != "O":
                if tag_name == current_label:
                    pass  # continue current span
                else:
                    if current_label is not None:
                        spans.append((current_label, current_start, i))
                    current_label = tag_name
                    current_start = i
            else:
                if current_label is not None:
                    spans.append((current_label, current_start, i))
                    current_label = None
        if current_label is not None:
            spans.append((current_label, current_start, len(tag_seq)))
        return spans
    pred_spans_all = defaultdict(set)
    gold_spans_all = defaultdict(set)
    for idx, (p_seq, g_seq) in enumerate(zip(all_pred_seqs, all_gold_seqs)):
        for label, s, e in flat_to_spans(p_seq):
            pred_spans_all[label].add((idx, s, e))
        for label, s, e in flat_to_spans(g_seq):
            gold_spans_all[label].add((idx, s, e))
    per_label = {}
    for label in LABELS:
        ps = pred_spans_all.get(label, set())
        gs = gold_spans_all.get(label, set())
        tp = len(ps & gs)
        fp = len(ps - gs)
        fn = len(gs - ps)
        p = tp / (tp + fp) if tp + fp > 0 else 0
        r = tp / (tp + fn) if tp + fn > 0 else 0
        f1 = 2 * p * r / (p + r) if p + r > 0 else 0
        per_label[label] = f1
    macro_f1 = sum(per_label.values()) / len(LABELS) if LABELS else 0
    return macro_f1, per_label

print("Training & evaluation functions defined")


## 12 - Prediction & Submission

In [ ]:
# ── Prediction & Submission ──
@torch.no_grad()
def predict_test(model, text_dict, tokenizer, cfg, device=DEVICE):
    model.eval()
    predictions = {}
    for doc_id in tqdm(text_dict, desc="Predicting"):
        text = get_doc_text(text_dict[doc_id])
        if not text.strip():
            predictions[doc_id] = []
            continue
        all_ids, all_spans = encode_with_offsets(tokenizer, text)
        n = len(all_ids)
        if n == 0:
            predictions[doc_id] = []
            continue
        step = cfg.window_stride if cfg.window_stride > 0 else cfg.window_size
        starts = list(range(0, max(1, n - cfg.window_size + 1), step))
        if starts[-1] + cfg.window_size < n:
            starts.append(max(0, n - cfg.window_size))
        token_logits = [None] * n
        token_dist   = [-1] * n
        for s in starts:
            e = min(s + cfg.window_size, n)
            w_ids = all_ids[s:e]
            pad_len = cfg.window_size - len(w_ids)
            inp = torch.tensor([w_ids + [0]*pad_len], dtype=torch.long, device=device)
            amask = torch.tensor([[1]*len(w_ids) + [0]*pad_len], dtype=torch.long, device=device)
            with torch.amp.autocast('cuda', enabled=cfg.use_amp):
                out = model(inp, attention_mask=amask)
            logits = out["logits"][0, :len(w_ids), :]
            for i_in_w in range(len(w_ids)):
                global_idx = s + i_in_w
                dist = min(i_in_w, len(w_ids) - 1 - i_in_w)
                if dist > token_dist[global_idx]:
                    token_dist[global_idx] = dist
                    token_logits[global_idx] = logits[i_in_w]
        # Get predicted tags
        tag_preds = []
        for i in range(n):
            if token_logits[i] is not None:
                tag_preds.append(token_logits[i].argmax().item())
            else:
                tag_preds.append(O_TAG)
        # Flat label decoding: merge consecutive same-label tokens into spans
        doc_spans = []
        current_label = None
        span_start_char = None
        span_end_char = None
        for i, tag_id in enumerate(tag_preds):
            tag_name = id2tag.get(tag_id, "O")
            cs, ce = all_spans[i]
            if tag_name != "O":
                if tag_name == current_label:
                    # Continue current span
                    span_end_char = max(span_end_char, ce) if span_end_char else ce
                else:
                    # New entity type — close previous span if any
                    if current_label is not None and span_start_char is not None:
                        doc_spans.append([span_start_char, span_end_char, current_label])
                    current_label = tag_name
                    span_start_char = cs
                    span_end_char = ce
            else:
                # O tag — close current span if any
                if current_label is not None and span_start_char is not None:
                    doc_spans.append([span_start_char, span_end_char, current_label])
                current_label = None
                span_start_char = None
                span_end_char = None
        # Close final span
        if current_label is not None and span_start_char is not None:
            doc_spans.append([span_start_char, span_end_char, current_label])
        # Trim boundary punctuation
        TRIM_CHARS = set('()[]{},:;/\\\'\"---!?.')
        trimmed = []
        for s_start, s_end, s_label in doc_spans:
            if s_start >= s_end:
                continue
            snippet = text[s_start:s_end]
            l = 0
            while l < len(snippet) and snippet[l] in TRIM_CHARS:
                l += 1
            r = len(snippet)
            while r > l and snippet[r - 1] in TRIM_CHARS:
                r -= 1
            if r > l:
                trimmed.append([s_start + l, s_start + r, s_label])
        predictions[doc_id] = trimmed
    return predictions

def save_submission(predictions, exp_name):
    pred_path = SUB_DIR / f"prediction_{exp_name}.json"
    zip_path  = SUB_DIR / f"submission_{exp_name}.zip"
    with open(pred_path, "w") as f:
        json.dump(predictions, f, ensure_ascii=False)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(pred_path, "prediction.json")
    n_ents = sum(len(v) for v in predictions.values())
    print(f"  {exp_name}: {len(predictions)} docs, {n_ents} entities -> {zip_path.name}")
    return zip_path

# Also save best checkpoint to Google Drive for other notebooks
def save_to_drive(exp_name):
    src = CKPT_DIR / f"{exp_name}_best.pt"
    if src.exists():
        dst_dir = os.path.join(DRIVE_ROOT, "ner_checkpoints")
        os.makedirs(dst_dir, exist_ok=True)
        shutil.copy(str(src), dst_dir)
        print(f"  Saved {exp_name}_best.pt to Google Drive")

print("Prediction & submission functions defined")


# Experiment 1: Full Fine-tuning
Unfreeze all backbone layers + NER head.

In [ ]:
set_seed(CFG.seed)
backbone_1 = load_phase1_backbone(CFG, PHASE1_CKPT, DEVICE)
model_1 = LegalNER(backbone_1, CFG.bio_num_tags, dropout=0.3)
for p in model_1.parameters():
    p.requires_grad = True

cfg_1 = copy.copy(CFG)
cfg_1.lr = 1e-5
cfg_1.num_epochs = 20
model_1 = train_ner(model_1, train_dl, val_dl, cfg_1, "exp1_full_ft", patience=7)

preds_1 = predict_test(model_1, fold3_text, tokenizer, CFG)
save_submission(preds_1, "exp1_full_ft")

# Save teacher checkpoint for Exp 6 (Knowledge Distillation)
TEACHER_CKPT = CKPT_DIR / "teacher_full_ft.pt"
torch.save(model_1.state_dict(), TEACHER_CKPT)
save_to_drive("exp1_full_ft")

# Also save teacher to Drive so Exp 6 notebook can load it
shutil.copy(str(TEACHER_CKPT), os.path.join(DRIVE_ROOT, "ner_checkpoints", "teacher_full_ft.pt"))
print("Teacher checkpoint saved to Drive for Exp 6")

del backbone_1
torch.cuda.empty_cache()


In [ ]:
# ── Download submission ──
from google.colab import files
for zf in sorted(SUB_DIR.glob("submission_*.zip")):
    print(f"Downloading {zf.name}...")
    files.download(str(zf))